# Install Dependencies

In [ ]:
# Install Hugging Face dependencies
!pip install -q transformers datasets

# Continued Pretraining on Domain-Specific Corpus

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling


# Load Model

In [ ]:
# ✅ Use a small model like DistilGPT-2
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # To avoid padding issues

model = AutoModelForCausalLM.from_pretrained(model_name)


# Load Dataset

In [ ]:
# ✅ Tiny medical dataset
texts = [
    "Hypertension is a chronic condition characterized by elevated blood pressure.",
    "Diabetes is caused by insufficient insulin production or response.",
    "MRI scans help visualize organs using magnetic fields and radio waves.",
    "An ECG records the electrical signals in the heart.",
    "Asthma involves inflammation and narrowing of the airways."
]

# Tokenize dataset for language modeling

In [ ]:
# Convert to Hugging Face Dataset
dataset = Dataset.from_dict({"text": texts})

# Tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=64)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Define training arguments

In [ ]:
# ✅ Training arguments (CPU-friendly)
training_args = TrainingArguments(
    output_dir="./distilgpt2-medical-pretrain",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    logging_steps=1,
    save_steps=10,
    save_total_limit=1,
    prediction_loss_only=True,
    report_to="none",  # Disable logging
)

# Train the Model

In [ ]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Train!
trainer.train()